## Краткое описание решения

### Признаки

События сначала фильтруются по правилу `window_start_ts <= event_ts < window_end_ts`, поэтому в признаки не попадает информация после окончания окна. Для каждой cookie строятся 68 признаков: возраст cookie и день недели, количество и разнообразие событий, абсолютные количества выбранных типов событий, три доли событий (`captcha_shown`, `favorite_add`, `contact_message_sent`), признаки платформы и User-Agent, длительность и интенсивность активности, интервалы между событиями, глубина поиска и pointer-признаки.

### Валидация

Поскольку test расположен хронологически после train, используется expanding window validation: `06–15 → 16.04`, `06–16 → 17.04`, `06–17 → 18.04`, `06–18 → 19.04`. На каждом фолде модель обучается только на прошлых датах и проверяется на следующем дне. Основная метрика — Precision при Recall не менее 0.7, рассчитанная функцией из `metric.py`. Средний результат CatBoost на четырёх фолдах — 0.7224, медианный — 0.6992.

### Модель

Сначала пробовал решать через лог регрессию, но получил плохое качество ~0.3(лог рег не работает с нелинейнным зависимостями), также обучал случайный лес по нему получил результаты кратно лучше, но предпочел бустинг ~0.62. Окончательно остановился на CatBoostClassifier для бинарной классификации. После валидации CatBoost обучается на всех размеченных cookie и создаёт `submission.csv` со score вероятности положительного класса. Пробовал оптимизировать через optuna но не получил серьезного увеличения в качестве.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from metric import precision_at_recall

SEED = 42
DATA_DIR = Path('data')

In [5]:
def _safe_div(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    '''Безопасно делит два ряда и возвращает 0 при нулевом знаменателе.'''
    return numerator.div(denominator.replace(0, np.nan)).fillna(0.0)

In [6]:
def build_features(meta: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    '''Агрегирует события внутри полуинтервала [window_start_ts, window_end_ts).'''
    metadata = meta[['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']].copy()

    # Присоединяем к событиям границы окна и оставляем только события внутри окна.
    window_events = events.merge(
        metadata[['cookie_id', 'window_start_ts', 'window_end_ts']],
        on='cookie_id', how='inner'
    )
    window_events = window_events[
        window_events['event_ts'].ge(window_events['window_start_ts'])
        & window_events['event_ts'].lt(window_events['window_end_ts'])
    ].copy()
    window_events['platform_norm'] = window_events['platform'].fillna('missing').str.lower()
    window_events['ua'] = window_events['user_agent'].fillna('').str.lower()
    window_events['event_name'] = window_events['event_name'].fillna('missing')

    metadata_by_cookie = metadata.set_index('cookie_id')
    feature_table = metadata_by_cookie[[]].copy()
    # Возраст cookie и день недели начала окна.
    feature_table['cookie_age_days'] = (
        metadata_by_cookie['window_start_ts'] - metadata_by_cookie['cookie_created_at']
    ).dt.total_seconds() / 86400
    feature_table['window_weekday'] = metadata_by_cookie['window_start_ts'].dt.weekday

    events_by_cookie = window_events.groupby('cookie_id', sort=False)
    # Интенсивность и разнообразие поведения.
    feature_table['n_events'] = events_by_cookie.size()
    feature_table['n_event_types'] = events_by_cookie['event_name'].nunique()
    feature_table['n_platforms'] = events_by_cookie['platform_norm'].nunique()
    feature_table['n_user_agents'] = events_by_cookie['ua'].nunique()
    for column in ['item_id', 'item_category', 'item_location', 'seller_type', 'search_query', 'search_page']:
        feature_table[f'nunique_{column}'] = events_by_cookie[column].nunique()

    # Абсолютные количества событий каждого типа.
    event_counts = pd.crosstab(window_events['cookie_id'], window_events['event_name']).add_prefix('event_')
    feature_table = feature_table.join(event_counts, how='left')
    # Доли выбранных типов событий от общего числа событий cookie.
    for event_name in ['captcha_shown', 'favorite_add', 'contact_message_sent']:
        count_name = f'event_{event_name}'
        rate_name = f'{count_name}_rate'
        if count_name in feature_table.columns:
            feature_table[rate_name] = _safe_div(feature_table[count_name], feature_table['n_events'])
        else:
            feature_table[rate_name] = 0.0

    # Платформа и укрупнённые признаки User-Agent.
    platform_counts = pd.crosstab(window_events['cookie_id'], window_events['platform_norm']).add_prefix('platform_')
    feature_table = feature_table.join(platform_counts, how='left')
    ua_patterns = {
        'headless': r'headless', 'python': r'python|requests|urllib',
        'scrapy': r'scrapy', 'curl': r'curl', 'node': r'node-fetch|node/',
        'go': r'go-http-client', 'selenium': r'selenium|webdriver|phantom',
        'bot_word': r'bot|spider|crawler', 'okhttp': r'okhttp',
        'mobile': r'mobile|android|iphone|ipad', 'linux': r'linux',
        'windows': r'windows', 'mac': r'macintosh|mac os',
    }
    for name, pattern in ua_patterns.items():
        is_match = window_events['ua'].str.contains(pattern, regex=True, na=False)
        feature_table[f'ua_{name}_events'] = is_match.groupby(window_events['cookie_id']).sum()
        feature_table[f'ua_{name}'] = is_match.groupby(window_events['cookie_id']).any().astype(int)

    # Длительность, интенсивность и регулярность активности.
    first_event = events_by_cookie['event_ts'].min()
    last_event = events_by_cookie['event_ts'].max()
    feature_table['active_seconds'] = (last_event - first_event).dt.total_seconds()
    feature_table['events_per_active_hour'] = _safe_div(feature_table['n_events'], feature_table['active_seconds'] / 3600)
    sorted_events = window_events.sort_values(['cookie_id', 'event_ts']).copy()
    interarrival_seconds = sorted_events.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    feature_table['mean_interarrival'] = interarrival_seconds.groupby(sorted_events['cookie_id']).mean()
    feature_table['median_interarrival'] = interarrival_seconds.groupby(sorted_events['cookie_id']).median()
    feature_table['std_interarrival'] = interarrival_seconds.groupby(sorted_events['cookie_id']).std()
    feature_table['min_interarrival'] = interarrival_seconds.groupby(sorted_events['cookie_id']).min()
    feature_table['n_fast_pairs'] = interarrival_seconds.le(2).groupby(sorted_events['cookie_id']).sum()
    feature_table['n_long_gaps'] = interarrival_seconds.ge(1800).groupby(sorted_events['cookie_id']).sum()

    # Глубина поиска и взаимодействие с указателем.
    for column, aggregation, suffix in [('search_page', 'max', 'max'), ('search_page', 'mean', 'mean'), ('pointer_x', 'std', 'std'), ('pointer_y', 'std', 'std')]:
        feature_table[f'{column}_{suffix}'] = events_by_cookie[column].agg(aggregation)
    feature_table['pointer_moves'] = events_by_cookie['pointer_x'].count()
    return feature_table.replace([np.inf, -np.inf], np.nan).fillna(0.0).reset_index()

In [7]:
def score_model(model, train_features, train_target, validation_features, validation_target, model_name):
    model.fit(train_features, train_target)
    scores = model.predict_proba(validation_features)[:, 1]
    value = precision_at_recall(validation_target, scores)
    print(f'{model_name}: P@R>=0.7={value:.4f}')
    return model, scores

def expanding_window_validation(feature_table, target, window_dates, feature_columns):
    '''Однодневная валидация с расширяющимся обучающим окном.'''
    dates = pd.to_datetime(window_dates).dt.normalize()
    rows = []
    for validation_day in sorted(dates.unique())[-4:]:
        train_idx = np.flatnonzero((dates < validation_day).to_numpy())
        validation_idx = np.flatnonzero((dates == validation_day).to_numpy())
        model = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05, loss_function='Logloss', eval_metric='AUC', random_seed=SEED, verbose=False, thread_count=4)
        model.fit(feature_table.iloc[train_idx][feature_columns], target[train_idx])
        scores = model.predict_proba(feature_table.iloc[validation_idx][feature_columns])[:, 1]
        rows.append({'validation_day': validation_day.date().isoformat(), 'train_rows': len(train_idx), 'validation_rows': len(validation_idx), 'validation_positives': int(target[validation_idx].sum()), 'catboost_p_at_r07': precision_at_recall(target[validation_idx], scores)})
    result = pd.DataFrame(rows)
    display(result)
    print('Среднее:', result['catboost_p_at_r07'].mean().round(4))
    print('Медиана:', result['catboost_p_at_r07'].median().round(4))
    return result

In [8]:
date_columns = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_columns)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_columns)
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['event_ts'])
metadata_all = pd.concat([train.drop(columns='target'), test], ignore_index=True)
feature_table_all = build_features(metadata_all, events).set_index('cookie_id')
train_features = feature_table_all.reindex(train.cookie_id).fillna(0)
test_features = feature_table_all.reindex(test.cookie_id).fillna(0)
target = train.target.to_numpy()
feature_columns = train_features.columns.tolist()
print('Количество признаков:', len(feature_columns))
print('Размеры:', train_features.shape, test_features.shape)

Количество признаков: 68
Размеры: (11091, 68) (4909, 68)


In [9]:
validation_results = expanding_window_validation(
    train_features, target, train.window_start_ts, feature_columns
)

,validation_day,train_rows,validation_rows,validation_positives,catboost_p_at_r07
0,2026-04-16,8400,740,60,0.636364
1,2026-04-17,9140,713,66,0.854545
2,2026-04-18,9853,606,52,0.684211
3,2026-04-19,10459,632,42,0.714286


Среднее: 0.7224
Медиана: 0.6992


In [10]:
final_model = CatBoostClassifier(
    iterations=500, depth=6, learning_rate=0.05,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=SEED, verbose=False, thread_count=4,
)
final_model.fit(train_features[feature_columns], target)
test_scores = final_model.predict_proba(test_features[feature_columns])[:, 1]
submission = pd.DataFrame({'cookie_id': test.cookie_id, 'score': np.clip(test_scores, 0, 1)})
assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert submission.score.between(0, 1).all()
submission.to_csv('submission.csv', index=False)
submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.000183
1,ck_a76ee3b3e3e522fd,0.139328
2,ck_94c9a4d382689e82,0.023327
3,ck_8eaf9509ad9462a0,0.001104
4,ck_9a88a5a989cb5bc6,0.010278
